## Скорость: python-движок против numba

Одинаковый результат, разное время. Первый вызов numba включает компиляцию.

In [5]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import time
import numpy as np
import polars as pl
import config
from src.signals import log_spread, rolling_zscore
from src.funding import load_funding, funding_on_bars
from src.engine import run_backtest
from src.engine_numba import run_backtest_fast

# ── данные, z, фандинг ──
df = pl.read_parquet(config.PROCESSED_DIR / "pair_5m.parquet")
z = rolling_zscore(log_spread(df["close_a"].to_numpy(), df["close_b"].to_numpy()), config.WINDOW)
tradable = ((df["n_traded_a"] > 0) & (df["n_traded_b"] > 0)).to_numpy()
open_time = df["open_time"].to_numpy()
fa = funding_on_bars(open_time, load_funding(config.SYMBOL_A))
fb = funding_on_bars(open_time, load_funding(config.SYMBOL_B))
args = (df["open_a"].to_numpy(), df["open_b"].to_numpy(),
        df["close_a"].to_numpy(), df["close_b"].to_numpy(), tradable, z)

# ── замер ──
t0 = time.perf_counter(); a = run_backtest(*args, "regular", fa, fb); t_py = time.perf_counter() - t0
t0 = time.perf_counter(); b = run_backtest_fast(*args, "regular", fa, fb); t_first = time.perf_counter() - t0
t0 = time.perf_counter(); b = run_backtest_fast(*args, "regular", fa, fb); t_fast = time.perf_counter() - t0

print(f"python:             {t_py:.2f} с")
print(f"numba, 1-й вызов:   {t_first:.2f} с  (с компиляцией)")
print(f"numba, дальше:      {t_fast:.3f} с  → быстрее в {t_py / t_fast:.0f} раз")
print("капитал совпадает:", np.allclose(a["equity"], b["equity"]))
print("сделок:", len(a["trades"]), "/", len(b["trades"]))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
python:             0.07 с
numba, 1-й вызов:   0.53 с  (с компиляцией)
numba, дальше:      0.001 с  → быстрее в 46 раз
капитал совпадает: True
сделок: 987 / 987
